<a href="https://colab.research.google.com/github/ImagingDataCommons/IDC-Tutorials/blob/master/notebooks/advanced_topics/image_visualization_with_ipyniivue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interactive 3D visualization of IDC images and segmentations with ipyniivue

---

## Summary

This notebook shows how to use [**ipyniivue**](https://github.com/niivue/ipyniivue) — the Jupyter
widget wrapper around the WebGL-based [NiiVue](https://github.com/niivue/niivue) medical image
viewer — to interactively visualize radiology images and segmentation overlays from
[NCI Imaging Data Commons (IDC)](https://imaging.datacommons.cancer.gov), directly inside a notebook.

Unlike a static screenshot, the embedded viewer lets you scroll through slices, pan/zoom, switch
between axial/coronal/sagittal/3D-render layouts, and toggle the segmentation overlay — all without
leaving the notebook and **without installing any desktop DICOM software**. Everything in this
notebook runs in [Google Colab](https://colab.research.google.com).

As an example we visualize a chest CT from the [National Lung Screening Trial (NLST)](https://imaging.datacommons.cancer.gov/explore/?filters_for_load=collection_id_nlst)
collection together with a whole-body multi-organ segmentation produced by
[TotalSegmentator](https://github.com/wasserth/TotalSegmentator) (one of the AI analysis results
hosted by IDC).

Upon completion of this tutorial, you will learn how to:
* find an image series in IDC that has an accompanying DICOM Segmentation (SEG) object
* download the image and segmentation with [`idc-index`](https://github.com/ImagingDataCommons/idc-index)
* convert a DICOM image series to NIfTI with [SimpleITK](https://simpleitk.org/), and a multi-segment
  DICOM SEG to a spatially-aligned label map with [highdicom](https://github.com/ImagingDataCommons/highdicom)
* build a color lookup table from the **recommended display colors stored inside the DICOM SEG itself**
* render the image volume in a multiplanar + 3D view with ipyniivue, overlay the labeled segmentation,
  and switch viewer layouts and overlay opacity

> **Why convert to NIfTI?** NiiVue renders volumetric data and aligns overlays using the spatial
> metadata (affine matrix) stored in each volume. Converting the DICOM image and the DICOM SEG to
> NIfTI while preserving their patient-coordinate geometry guarantees that the segmentation lands
> exactly on top of the anatomy it describes.

---
Initial version: Jun 2026

Updated: Jun 2026

## 1. Setup

Install the packages used in this notebook. In Google Colab `idc-index`, `SimpleITK`, `highdicom`,
and `ipyniivue` are not pre-installed, so we add them here. This takes a minute or so.

In [ ]:
%%capture
%pip install --upgrade idc-index ipyniivue highdicom SimpleITK

### Enable interactive widgets in Colab

`ipyniivue` is a custom Jupyter widget. Google Colab does not render third-party widgets unless the
*custom widget manager* is explicitly enabled, so we do that here. The same code is a harmless no-op
when the notebook runs in a local Jupyter/JupyterLab environment.

In [ ]:
try:
    # Running in Google Colab: enable rendering of third-party (anywidget-based) widgets
    from google.colab import output
    output.enable_custom_widget_manager()
    IN_COLAB = True
    print("Running in Google Colab - custom widget manager enabled.")
except ImportError:
    IN_COLAB = False
    print("Not running in Colab (e.g. local Jupyter) - no extra widget setup needed.")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk
import highdicom as hd

from idc_index import IDCClient
from ipyniivue import NiiVue, SliceType

client = IDCClient()
print("idc-index is using IDC data version:", client.get_idc_version())

## 2. Find an image series that has a segmentation

IDC stores segmentations as standard **DICOM Segmentation (SEG)** objects. The `seg_index` table
records, for every SEG series, the `SeriesInstanceUID` of the original image series it was derived
from (`segmented_SeriesInstanceUID`). We join it back to the main `index` to find image+SEG pairs.

For this tutorial we use a chest CT from the **NLST** collection (license: CC BY, commercial use
allowed) together with its **TotalSegmentator** whole-body organ segmentation, which is published in
IDC as the `TotalSegmentator-CT-Segmentations` analysis result. The query below lists candidate
pairs; we then pin one specific example by its UIDs so the notebook is fully reproducible.

In [ ]:
client.fetch_index("seg_index")

candidates = client.sql_query("""
    SELECT
        seg.SeriesInstanceUID                AS seg_series_uid,
        seg.segmented_SeriesInstanceUID      AS image_series_uid,
        img.collection_id,
        seg_meta.SeriesDescription           AS seg_description,
        ROUND(img.series_size_MB, 1)         AS image_MB,
        ROUND(seg_meta.series_size_MB, 1)    AS seg_MB
    FROM seg_index seg
    JOIN index img      ON seg.segmented_SeriesInstanceUID = img.SeriesInstanceUID
    JOIN index seg_meta ON seg.SeriesInstanceUID           = seg_meta.SeriesInstanceUID
    WHERE seg_meta.analysis_result_id = 'TotalSegmentator-CT-Segmentations'
      AND img.collection_id = 'nlst'
    ORDER BY image_MB
    LIMIT 5
""")
candidates

In [ ]:
# Pin one specific example for reproducibility (a chest CT + its TotalSegmentator multi-organ SEG)
image_series_uid = "1.2.840.113654.2.55.71041873368734986406977154644223539362"
seg_series_uid   = "1.2.276.0.7230010.3.1.3.313263360.84.1706324554.366745"

## 3. Download the image and segmentation

We download each series into its own folder. `dirTemplate=""` puts the DICOM files directly in the
target directory (no nested `collection/patient/study/...` hierarchy), which keeps the paths simple
for the conversion step below.

In [ ]:
work_dir = Path("idc_niivue_demo")
image_dir = work_dir / "image_dicom"
seg_dir   = work_dir / "seg_dicom"

client.download_dicom_series(seriesInstanceUID=image_series_uid,
                             downloadDir=str(image_dir), dirTemplate="")
client.download_dicom_series(seriesInstanceUID=seg_series_uid,
                             downloadDir=str(seg_dir), dirTemplate="")

print("Image DICOM files:", len(list(image_dir.glob("*.dcm"))))
print("SEG DICOM files:  ", len(list(seg_dir.glob("*.dcm"))))

## 4. Convert the DICOM image series to NIfTI

A CT/MR series is a stack of single-slice DICOM files. `SimpleITK` reads the whole stack, sorts the
slices into a 3D volume, and writes a single NIfTI file. SimpleITK preserves the patient-coordinate
geometry (origin, voxel spacing, orientation) and handles the DICOM (LPS) to NIfTI (RAS) coordinate
convention for us.

In [ ]:
reader = sitk.ImageSeriesReader()
reader.SetFileNames(reader.GetGDCMSeriesFileNames(str(image_dir)))
image = reader.Execute()

image_nii = work_dir / "image.nii.gz"
sitk.WriteImage(image, str(image_nii))

print("Volume size (cols, rows, slices):", image.GetSize())
print("Voxel spacing (mm):", tuple(round(s, 3) for s in image.GetSpacing()))
print("Saved:", image_nii)

## 5. Convert the DICOM SEG to an aligned NIfTI label map

`highdicom` reads the DICOM SEG and reconstructs the segmentation as a 3D volume that carries the
same frame-of-reference geometry as the source image. `get_volume(combine_segments=True)` produces a
single **label map** where background = 0 and each organ gets its own integer label.

We save it as NIfTI, converting highdicom's index&rarr;LPS affine to the NIfTI index&rarr;RAS
convention by flipping the first two axes (`diag(-1, -1, 1, 1)`). Because both the image and the
label map are written in the same RAS world coordinates, NiiVue overlays them in perfect alignment.

In [ ]:
seg = hd.seg.segread(next(seg_dir.glob("*.dcm")))

# Map each segment number (the integer voxel value in the label map) to its label
segment_labels = {n: seg.get_segment_description(n).segment_label for n in seg.segment_numbers}
print(f"This SEG object contains {len(segment_labels)} segments, e.g.:")
for n in list(segment_labels)[:8]:
    print(f"  {n:>3}: {segment_labels[n]}")
print("  ...")

In [ ]:
# Combined multi-label map: 0 = background, 1..N = organs. Some TotalSegmentator structures can
# touch/overlap, so we skip the overlap check (higher-numbered label wins on any overlap).
vol = seg.get_volume(combine_segments=True, skip_overlap_checks=True)

# LPS (DICOM) -> RAS (NIfTI): flip the x and y axes
LPS_TO_RAS = np.diag([-1.0, -1.0, 1.0, 1.0])

labelmap_nii = work_dir / "segmentation_labelmap.nii.gz"
nib.save(nib.Nifti1Image(vol.array.astype(np.uint8), LPS_TO_RAS @ vol.affine), str(labelmap_nii))
print("Label map shape:", vol.array.shape, "| labels present:", int(vol.array.max()))

### Use the colors stored inside the DICOM SEG

A DICOM SEG can carry a **Recommended Display CIELab Value** for each segment — the color the data
producer intended for that structure. Rather than inventing our own palette, we read these colors
straight from the SEG and build a NiiVue *label colormap* (a lookup table mapping each integer label
to an RGBA color and a name).

The recommended color is stored in the DICOM CIELab encoding (PS3.3 C.10.7.1.1), so we decode it to
CIELab and then convert to sRGB.

In [ ]:
def dicom_cielab_to_rgb(cielab):
    """Convert a DICOM-encoded Recommended Display CIELab Value (3 uint16) to an sRGB [0-255] triplet."""
    L = cielab[0] / 65535.0 * 100.0          # L*  in [0, 100]
    a = cielab[1] / 65535.0 * 255.0 - 128.0  # a*  in [-128, 127]
    b = cielab[2] / 65535.0 * 255.0 - 128.0  # b*  in [-128, 127]

    # CIELab -> XYZ (D65 white point)
    fy = (L + 16) / 116.0
    fx = fy + a / 500.0
    fz = fy - b / 200.0
    finv = lambda t: t ** 3 if t ** 3 > 0.008856 else (t - 16 / 116.0) / 7.787
    X = 95.047 * finv(fx) / 100.0
    Y = 100.0 * finv(fy) / 100.0
    Z = 108.883 * finv(fz) / 100.0

    # XYZ -> linear sRGB -> gamma-corrected sRGB
    r = 3.2406 * X - 1.5372 * Y - 0.4986 * Z
    g = -0.9689 * X + 1.8758 * Y + 0.0415 * Z
    bl = 0.0557 * X - 0.2040 * Y + 1.0570 * Z

    def gamma(c):
        c = min(max(c, 0.0), 1.0)
        return 1.055 * c ** (1 / 2.4) - 0.055 if c > 0.0031308 else 12.92 * c

    return [int(round(gamma(c) * 255)) for c in (r, g, bl)]


# Build a NiiVue label colormap. Entry 0 is background (fully transparent); entries 1..N use the
# SEG's own recommended display color for each organ.
R, G, B, A, I, labels = [0], [0], [0], [0], [0], ["background"]
for n in seg.segment_numbers:
    desc = seg.get_segment_description(n)
    rgb = dicom_cielab_to_rgb(desc.RecommendedDisplayCIELabValue)
    R.append(rgb[0]); G.append(rgb[1]); B.append(rgb[2]); A.append(255)
    I.append(int(n)); labels.append(desc.segment_label)

label_colormap = {"R": R, "G": G, "B": B, "A": A, "I": I, "labels": labels}
print(f"Built a label colormap with {len(R) - 1} organ colors taken from the DICOM SEG.")

Let's preview a few of the organ colors that came from the SEG:

In [ ]:
legend = pd.DataFrame({
    "label": labels[1:],
    "hex":   [f"#{r:02x}{g:02x}{b:02x}" for r, g, b in zip(R[1:], G[1:], B[1:])],
})
legend.head(12).style.apply(
    lambda row: [f"background-color: {row['hex']}; color: white"] * len(row), axis=1
)

## 6. Visualize the image with ipyniivue

We start with the image volume alone. Creating a `NiiVue` widget and calling `load_volumes` with a
list of `{"path": ...}` dictionaries renders the volume. By default NiiVue shows a **multiplanar**
layout (axial, coronal, sagittal, plus a 3D render).

The `cal_min`/`cal_max` values set the display window in the image's native intensity units. For CT
these are Hounsfield Units; `-1000` to `400` gives a good lung/soft-tissue window.

> Scroll with the mouse wheel to move through slices, drag to pan, and use the render panel to rotate
> the 3D view. If you don't see the viewer in Colab, make sure the "Enable interactive widgets" cell
> above ran successfully.

In [ ]:
nv = NiiVue(height=500)
nv.load_volumes([
    {"path": str(image_nii), "colormap": "gray", "cal_min": -1000, "cal_max": 400},
])
nv

## 7. Overlay the multi-organ segmentation

Now we load the label map on top of the image and apply the label colormap we built from the SEG.
The image is layer 0 and the segmentation is layer 1; we set the overlay opacity in `load_volumes`
and then attach the discrete label colormap with `set_colormap_label`, so each organ is drawn in its
own DICOM-recommended color. Hovering over the overlay in the viewer shows the organ name.

In [ ]:
nv_overlay = NiiVue(height=500)
nv_overlay.load_volumes([
    {"path": str(image_nii),    "colormap": "gray", "cal_min": -1000, "cal_max": 400},
    {"path": str(labelmap_nii), "opacity": 0.5},
])

# Apply the per-label colors taken from the DICOM SEG to the overlay (layer index 1)
nv_overlay.volumes[1].set_colormap_label(label_colormap)
nv_overlay

## 8. Customize the view

The viewer is fully scriptable from Python. A few common adjustments:

* **`set_slice_type`** switches the layout: `SliceType.AXIAL`, `CORONAL`, `SAGITTAL`, `RENDER` (3D
  only), or `SliceType.MULTIPLANAR` (the default 2x2 grid).
* **`set_opacity(index, value)`** changes the opacity of the volume at a given layer index
  (0 = base image, 1 = the segmentation overlay), letting you fade the segmentation in and out.

Run the cell below and watch the viewer above update live.

In [ ]:
# Show only the axial plane, and make the segmentation overlay more transparent
nv_overlay.set_slice_type(SliceType.AXIAL)
nv_overlay.set_opacity(1, 0.3)   # layer 1 == the segmentation overlay

Switch back to the multiplanar + 3D layout and a more opaque overlay:

In [ ]:
nv_overlay.set_slice_type(SliceType.MULTIPLANAR)
nv_overlay.set_opacity(1, 0.6)

## 9. Cite the data you used

Most IDC data (including this collection) is covered by a CC BY license, which permits commercial use
**with attribution**. `idc-index` can generate ready-to-use citations for any selection of data —
here both the NLST images and the TotalSegmentator analysis result.

In [ ]:
citations = client.citations_from_selection(seriesInstanceUID=[image_series_uid, seg_series_uid])
for c in citations:
    print(c, "\n")

## 10. Clean up (optional)

Remove the downloaded DICOM and converted NIfTI files to free disk space.

In [ ]:
import shutil
# shutil.rmtree(work_dir, ignore_errors=True)   # uncomment to delete the downloaded/converted files
print("To clean up, uncomment the line above and re-run this cell.")

## Next steps

* Try a different image — the query in Section&nbsp;2 works for any `collection_id` with SEG objects;
  drop the `analysis_result_id` filter to also include expert/manual segmentations. Browse collections
  in the [IDC Portal](https://portal.imaging.datacommons.cancer.gov/explore/).
* ipyniivue can also display **surface meshes** and **multi-frame 4D** data, draw/edit segmentations
  in-browser, and export screenshots — see the [ipyniivue documentation](https://niivue.github.io/ipyniivue/).
* For a zero-setup browser viewer (no conversion needed), `client.get_viewer_URL(seriesInstanceUID=...)`
  returns an OHIF/Slim viewer link for any IDC series.
* More tutorials are available in the
  [IDC-Tutorials repository](https://github.com/ImagingDataCommons/IDC-Tutorials).

## Support

If you have any questions about this notebook, please post your question on the
[IDC User Forum](https://discourse.canceridc.dev) or
[open an issue](https://github.com/ImagingDataCommons/IDC-Tutorials/issues/new) in the
[IDC Tutorials repository](https://github.com/ImagingDataCommons/IDC-Tutorials).

## Acknowledgments

Imaging Data Commons has been funded in whole or in part with Federal funds from the National Cancer
Institute, National Institutes of Health, under Task Order No. HHSN26110071 under Contract No.
HHSN261201500003I.

If you use IDC in your research, please cite the following publication:

> Fedorov, A., Longabaugh, W. J. R., Pot, D., Clunie, D. A., Pieper, S. D., Gibbs, D. L., Bridge, C., Herrmann, M. D., Homeyer, A., Lewis, R., Aerts, H. J. W., Krishnaswamy, D., Thiriveedhi, V. K., Ciausu, C., Schacherer, D. P., Bontempi, D., Pihl, T., Wagner, U., Farahani, K., Kim, E. & Kikinis, R. _National Cancer Institute Imaging Data Commons: Toward Transparency, Reproducibility, and Scalability in Imaging Artificial Intelligence_. RadioGraphics (2023). https://doi.org/10.1148/rg.230180